[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab02.ipynb)

# Bab 2 — Lingkungan Kerja dan Data Medis
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4** (*Runtime → Change runtime type → T4 GPU*), lalu **Runtime → Run all**.

Isi:
1. Lingkungan kerja: CPU vs GPU (perkalian matriks), RAM, disk, versi paket.
2. **DICOM sintetis** dengan `pydicom` dari citra MedMNIST (X-ray PneumoniaMNIST dan irisan OrganMNIST3D): tag, *pixel data*, *rescale* dan *window/level*.
3. Perbandingan format: DICOM 12 bit vs PNG 16 bit vs JPEG 8 bit.
4. **NIfTI** dengan `nibabel` dari volume OrganMNIST3D: *affine*, *pixdim*, orientasi, irisan ortogonal.
5. **Anonimisasi DICOM** (subset Basic Application Confidentiality Profile, DICOM PS3.15 Lampiran E) pada data yang **SELURUHNYA FIKTIF**.
6. **Demonstrasi kebocoran data** pada NIH Malaria Cell Images: pembagian acak per sel vs pembagian per kelompok-pasien.
7. Reproduktibilitas: seed, uji pengulangan, daftar versi paket, kartu model.

Semua hasil disimpan ke `/content/hasil_bab02/` dan dizip menjadi `/content/hasil_bab02.zip`.

> **Penting.** Semua identitas pasien di notebook ini **FIKTIF** (dibuat-buat untuk latihan). Citra MedMNIST dan NIH Malaria sudah dianonimkan oleh penyedianya.
> Nilai "HU" pada contoh CT adalah **HU semu** hasil pemetaan linear untuk latihan, **bukan** nilai HU asli pasien.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`
Sama di semua notebook buku. Dataset disimpan permanen di `MyDrive/Buku Visi Komputer Kesehatan/dataset/<nama>/` dan hanya diunduh dari sumber resmi bila belum ada atau rusak.

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 2.0 Lingkungan dan pencatat waktu

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB02_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB02_OUT", "/content/hasil_bab02"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

Paket: `pydicom` biasanya belum ada di Colab, jadi dipasang di sini. `nibabel` umumnya sudah ada; bila belum, ikut dipasang.

In [ ]:
with catat_waktu("1_pip_install"):
    import importlib.util
    paket = ["medmnist==3.0.2", "pydicom>=3.0,<4"]
    if importlib.util.find_spec("nibabel") is None: paket.append("nibabel")
    jalankan("pip install -q " + " ".join(f'"{p}"' for p in paket))

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; import matplotlib.pyplot as plt
import pydicom, nibabel as nib, PIL, sklearn, medmnist
from PIL import Image, ImageDraw
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import generate_uid, ExplicitVRLittleEndian, SecondaryCaptureImageStorage
from pydicom.pixels import apply_modality_lut, apply_voi_lut
from sklearn.metrics import roc_auc_score, accuracy_score
from medmnist import INFO
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
         "pydicom": pydicom.__version__, "nibabel": nib.__version__, "medmnist": medmnist.__version__,
         "scikit-learn": sklearn.__version__, "pillow": PIL.__version__, "matplotlib": matplotlib.__version__,
         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))

SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()

## 2.1 CPU vs GPU: mengapa runtime GPU penting
Perkalian matriks persegi float32 diulang beberapa kali; waktu dicatat setelah sinkronisasi GPU. Angka bergantung pada mesin Colab yang diperoleh.

In [ ]:
def ukur_matmul(n, dev, ulang=5):
    a = torch.randn(n, n, device=dev); b = torch.randn(n, n, device=dev)
    (a @ b); torch.cuda.synchronize() if dev.type == "cuda" else None   # pemanasan
    t0 = time.perf_counter()
    for _ in range(ulang): c = a @ b
    torch.cuda.synchronize() if dev.type == "cuda" else None
    dt = (time.perf_counter() - t0) / ulang
    return dt, 2 * n**3 / dt / 1e9
baris = []
with catat_waktu("2_cpu_vs_gpu"):
    for n in ([256, 512] if SMOKE else [1024, 2048, 4096]):
        for dev in [torch.device("cpu")] + ([torch.device("cuda")] if torch.cuda.is_available() else []):
            dt, gf = ukur_matmul(n, dev)
            baris.append({"ukuran_matriks": n, "perangkat": dev.type, "detik_per_matmul": dt, "GFLOPS": gf})
T21 = pd.DataFrame(baris); T21.to_csv(OUT / "tabel_2_1_cpu_vs_gpu.csv", index=False); T21

## 2.2 Membuat DICOM sintetis dengan `pydicom`
Citra MedMNIST (sudah dianonimkan, 8 bit) dibungkus menjadi objek DICOM *Secondary Capture*:
- **X-ray**: PneumoniaMNIST 128 px, diperbesar ke 256 px dengan interpolasi bilinear dan disimpan 12 bit (0–4080) agar mirip radiograf, dengan *WindowCenter/WindowWidth*. Karena sumbernya 8 bit, detail ekstra di 12 bit hanya berasal dari interpolasi.
- **CT**: satu irisan aksial OrganMNIST3D (28³); nilai 0–255 dipetakan linear ke **HU semu** −1000…+1000 lewat *RescaleSlope/RescaleIntercept*. Ini **ilustrasi**, bukan HU asli.
Identitas pasien diisi `ANONIM` (tanpa data pribadi).

In [ ]:
DIR_DCM = OUT / "dicom_sintetis"; DIR_DCM.mkdir(exist_ok=True)
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn])
    return np.load(folder / fn)

def buat_dicom(arr, path, modality, judul, slope=1.0, intercept=0.0, wc=None, ww=None, bits=12, identitas=None, tambahan=None):
    meta = FileMetaDataset()
    meta.MediaStorageSOPClassUID = SecondaryCaptureImageStorage
    meta.MediaStorageSOPInstanceUID = generate_uid(); meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds = Dataset(); ds.file_meta = meta
    ds.SOPClassUID = meta.MediaStorageSOPClassUID; ds.SOPInstanceUID = meta.MediaStorageSOPInstanceUID
    ds.StudyInstanceUID = generate_uid(); ds.SeriesInstanceUID = generate_uid()
    ds.Modality = modality; ds.ConversionType = "WSD"; ds.SeriesDescription = judul
    idt = identitas or {"PatientName": "ANONIM", "PatientID": "ANONIM-0001"}
    for k, v in idt.items(): setattr(ds, k, v)
    ds.Rows, ds.Columns = arr.shape; ds.SamplesPerPixel = 1; ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = 16; ds.BitsStored = bits; ds.HighBit = bits - 1; ds.PixelRepresentation = 0
    ds.PixelSpacing = [1.0, 1.0]           # nilai ilustratif, bukan ukuran fisik asli
    ds.RescaleSlope = slope; ds.RescaleIntercept = intercept; ds.RescaleType = "US" if intercept == 0 else "HU"
    if wc is not None: ds.WindowCenter = wc; ds.WindowWidth = ww
    for k, v in (tambahan or {}).items(): setattr(ds, k, v)
    ds.PixelData = arr.astype(np.uint16).tobytes()
    ds.save_as(path, enforce_file_format=True)
    return path

with catat_waktu("3_dicom_sintetis"):
    pn = npz_medmnist("pneumoniamnist", 128)
    xr8 = pn["test_images"][int(np.where(pn["test_labels"].reshape(-1) == 0)[0][0])]   # citra uji pertama berlabel normal
    # perbesar 128->256 dengan interpolasi bilinear pada nilai float lalu simpan 12 bit (0..4080): nilai antara tetap terjaga
    xr12 = np.clip(np.round(np.asarray(Image.fromarray(xr8.astype(np.float32)).resize((256, 256), Image.BILINEAR)) * 16), 0, 4080).astype(np.uint16)
    p_xr = buat_dicom(xr12, DIR_DCM / "xray_sintetis.dcm", "DX", "X-ray sintetis dari PneumoniaMNIST", wc=2040, ww=4080)
    o3 = npz_medmnist("organmnist3d", 28)
    vol = o3["test_images"][0].astype(np.float32)            # (28,28,28), nilai 0..1 atau 0..255
    vol8 = (vol / vol.max() * 255).astype(np.uint8) if vol.max() > 0 else vol.astype(np.uint8)
    irisan = vol8[vol8.shape[0] // 2]
    ct_px = (irisan.astype(np.float32) / 255 * 2000).round().astype(np.uint16)   # 0..2000 -> HU semu = px - 1000
    p_ct = buat_dicom(ct_px, DIR_DCM / "ct_sintetis.dcm", "CT", "CT sintetis (HU semu) dari OrganMNIST3D",
                      slope=1.0, intercept=-1000.0, wc=40, ww=400)
    print("rentang nilai volume OrganMNIST3D asli:", float(vol.min()), float(vol.max()), vol.dtype)

def tabel_tag(ds, maks=60):
    rows = []
    for el in ds.file_meta: rows.append({"tag": str(el.tag), "keyword": el.keyword, "VR": el.VR, "nilai": str(el.value)[:60]})
    for el in ds:
        if el.keyword == "PixelData": rows.append({"tag": str(el.tag), "keyword": el.keyword, "VR": el.VR, "nilai": f"<{len(el.value)} byte>"})
        else: rows.append({"tag": str(el.tag), "keyword": el.keyword, "VR": el.VR, "nilai": str(el.value)[:60]})
    return pd.DataFrame(rows[:maks])
ds_xr = pydicom.dcmread(p_xr); ds_ct = pydicom.dcmread(p_ct)
T22 = tabel_tag(ds_xr); T22.to_csv(OUT / "tabel_2_2_tag_dicom_xray.csv", index=False)
tabel_tag(ds_ct).to_csv(OUT / "tabel_2_3_tag_dicom_ct.csv", index=False)
T22

## 2.3 *Rescale* dan *window/level*
`apply_modality_lut` mengubah nilai piksel tersimpan menjadi nilai fisik (HU semu); *window/level* memilih rentang yang ditampilkan.
Jendela umum: paru (L −600, W 1500), jaringan lunak (L 40, W 400), tulang (L 400, W 1800).

In [ ]:
def jendela(hu, L, W):
    lo, hi = L - W / 2, L + W / 2
    return np.clip((hu - lo) / (hi - lo), 0, 1)
hu = apply_modality_lut(ds_ct.pixel_array, ds_ct).astype(np.float32)
xr_voi = apply_voi_lut(ds_xr.pixel_array, ds_xr).astype(np.float32)
fig, ax = plt.subplots(1, 5, figsize=(13, 3))
ax[0].imshow(xr_voi, cmap="gray"); ax[0].set_title("X-ray DICOM 12 bit\n(WC 2040 / WW 4080)")
ax[1].imshow(hu, cmap="gray"); ax[1].set_title(f"CT: HU semu mentah\n[{hu.min():.0f}, {hu.max():.0f}]")
for a, (nama, L, W) in zip(ax[2:], [("paru", -600, 1500), ("jaringan lunak", 40, 400), ("tulang", 400, 1800)]):
    a.imshow(jendela(hu, L, W), cmap="gray", vmin=0, vmax=1); a.set_title(f"jendela {nama}\nL {L} / W {W}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "gambar_2_1_dicom_window.png", bbox_inches="tight"); plt.show()

## 2.4 Format penyimpanan: DICOM vs PNG 16 bit vs JPEG 8 bit

In [ ]:
DIR_FMT = OUT / "format"; DIR_FMT.mkdir(exist_ok=True)
Image.fromarray(xr12.astype(np.uint16)).save(DIR_FMT / "xray_16bit.png")
xr_as8 = (xr12 / 4080 * 255).round().astype(np.uint8)
Image.fromarray(xr_as8).save(DIR_FMT / "xray_8bit.png")
Image.fromarray(xr_as8).save(DIR_FMT / "xray_q90.jpg", quality=90)
baris = []
for nama, p in [("DICOM (12 bit dalam 16 bit)", p_xr), ("PNG 16 bit (lossless)", DIR_FMT / "xray_16bit.png"),
                ("PNG 8 bit (kuantisasi)", DIR_FMT / "xray_8bit.png"), ("JPEG 8 bit q=90 (lossy)", DIR_FMT / "xray_q90.jpg")]:
    if str(p).endswith(".dcm"): arr = pydicom.dcmread(p).pixel_array.astype(np.float64)
    else: arr = np.array(Image.open(p)).astype(np.float64)
    if arr.max() <= 255 and not str(p).endswith(".dcm") and "16" not in nama: arr = arr / 255 * 4080
    err = np.abs(arr - xr12.astype(np.float64))
    baris.append({"format": nama, "ukuran_byte": Path(p).stat().st_size, "nilai_unik": int(len(np.unique(arr))),
                  "galat_maks_vs_asli_12bit": float(err.max()), "galat_rerata": float(err.mean()), "metadata_klinis": "ya" if str(p).endswith(".dcm") else "tidak"})
T24 = pd.DataFrame(baris); T24.to_csv(OUT / "tabel_2_4_format.csv", index=False); T24

## 2.5 NIfTI dengan `nibabel`
Volume OrganMNIST3D (28³) disimpan sebagai `.nii.gz` dengan *affine* diagonal (spasi voksel **ilustratif** 2,0 × 2,0 × 2,5 mm; MedMNIST tidak menyimpan spasi asli).

In [ ]:
with catat_waktu("4_nifti"):
    spasi = (2.0, 2.0, 2.5)
    aff = np.diag([*spasi, 1.0]); aff[:3, 3] = [-spasi[i] * vol8.shape[i] / 2 for i in range(3)]   # pusat volume di (0,0,0)
    img = nib.Nifti1Image(np.transpose(vol8, (2, 1, 0)).astype(np.int16), aff)   # MedMNIST (z,y,x) -> NIfTI (x,y,z)
    img.header.set_xyzt_units("mm"); img.set_qform(aff, code=1); img.set_sform(aff, code=1)
    p_nii = OUT / "organ_sintetis.nii.gz"; nib.save(img, p_nii)
    im2 = nib.load(p_nii); h = im2.header; data = np.asanyarray(im2.dataobj)
    vox = np.array([14, 14, 14]); dunia = nib.affines.apply_affine(im2.affine, vox)
    T25 = pd.DataFrame([
        {"field": "shape", "nilai": str(data.shape)}, {"field": "datatype", "nilai": str(h.get_data_dtype())},
        {"field": "pixdim (mm)", "nilai": str(tuple(float(v) for v in h.get_zooms()))},
        {"field": "orientasi (aff2axcodes)", "nilai": "".join(nib.aff2axcodes(im2.affine))},
        {"field": "qform_code / sform_code", "nilai": f"{int(h['qform_code'])} / {int(h['sform_code'])}"},
        {"field": "affine baris 1", "nilai": str(np.round(im2.affine[0], 2).tolist())},
        {"field": "affine baris 2", "nilai": str(np.round(im2.affine[1], 2).tolist())},
        {"field": "affine baris 3", "nilai": str(np.round(im2.affine[2], 2).tolist())},
        {"field": "voksel (14,14,14) -> dunia (mm)", "nilai": str(np.round(dunia, 2).tolist())},
        {"field": "ukuran berkas (byte)", "nilai": str(p_nii.stat().st_size)}])
    T25.to_csv(OUT / "tabel_2_5_nifti_header.csv", index=False)
    zx, zy, zz = h.get_zooms()
    fig, ax = plt.subplots(1, 3, figsize=(9, 3.2))
    ax[0].imshow(data[:, :, data.shape[2] // 2].T, cmap="gray", origin="lower", aspect=zy / zx); ax[0].set_title("aksial (x–y)")
    ax[1].imshow(data[:, data.shape[1] // 2, :].T, cmap="gray", origin="lower", aspect=zz / zx); ax[1].set_title("koronal (x–z)")
    ax[2].imshow(data[data.shape[0] // 2, :, :].T, cmap="gray", origin="lower", aspect=zz / zy); ax[2].set_title("sagital (y–z)")
    for a in ax: a.axis("off")
    plt.suptitle("Irisan ortogonal volume NIfTI (OrganMNIST3D, spasi ilustratif)"); plt.tight_layout()
    plt.savefig(OUT / "gambar_2_2_nifti_ortogonal.png", bbox_inches="tight"); plt.show()
T25

## 2.6 Anonimisasi DICOM (data FIKTIF)
Kita membuat satu DICOM yang sengaja diisi **identitas FIKTIF** (nama, ID, tanggal lahir, institusi, dokter perujuk), sebuah *private tag*,
dan teks identitas FIKTIF yang "tertanam" pada piksel (*burned-in annotation*). Lalu diterapkan subset aturan **Basic Application Confidentiality
Profile** (DICOM PS3.15 Lampiran E, Tabel E.1-1): **Z** = kosongkan, **X** = hapus, **U** = ganti UID, ditambah penghapusan *private tag*,
penandaan `PatientIdentityRemoved = YES`, dan penutupan wilayah teks pada piksel. Daftar aturan ini **tidak lengkap** — untuk data nyata gunakan
profil penuh dan alat yang tervalidasi.

In [ ]:
IDENTITAS_FIKTIF = {"PatientName": "FIKTIF^CONTOH", "PatientID": "FIKTIF-000123", "PatientBirthDate": "19800101",
                    "PatientSex": "O", "InstitutionName": "RUMAH SAKIT FIKTIF", "ReferringPhysicianName": "FIKTIF^DOKTER",
                    "AccessionNumber": "FIKTIF-ACC-1", "StudyDate": "20260101", "StudyTime": "101500"}
with catat_waktu("5_anonimisasi"):
    besar = np.array(Image.fromarray(xr8).resize((512, 512), Image.BILINEAR))
    kanvas = Image.fromarray(besar); dr = ImageDraw.Draw(kanvas)
    KOTAK = (8, 8, 330, 52)
    dr.rectangle(KOTAK, fill=0)
    dr.text((14, 12), "FIKTIF^CONTOH  ID FIKTIF-000123", fill=255); dr.text((14, 30), "LAHIR 1980-01-01  RS FIKTIF", fill=255)
    px_phi = (np.array(kanvas).astype(np.uint16) * 16)
    p_phi = buat_dicom(px_phi, DIR_DCM / "fiktif_sebelum.dcm", "DX", "Contoh dengan identitas FIKTIF", wc=2040, ww=4080,
                       identitas=IDENTITAS_FIKTIF, tambahan={"BurnedInAnnotation": "YES"})
    ds = pydicom.dcmread(p_phi)
    ds.private_block(0x0011, "CONTOH_VENDOR_FIKTIF", create=True).add_new(0x01, "LO", "catatan internal FIKTIF")
    ds.save_as(p_phi)

    ATURAN = {"PatientName": "Z", "PatientID": "Z", "PatientBirthDate": "Z", "PatientSex": "Z", "ReferringPhysicianName": "Z",
              "AccessionNumber": "Z", "StudyDate": "Z", "StudyTime": "Z", "InstitutionName": "X", "SeriesDescription": "X",
              "StudyInstanceUID": "U", "SeriesInstanceUID": "U", "SOPInstanceUID": "U"}
    def anonimkan(ds, kotak_teks=None):
        import copy; ds = copy.deepcopy(ds)   # salinan, aslinya tidak diubah
        log = []
        for kw, aksi in ATURAN.items():
            if kw not in ds: continue
            lama = str(ds.data_element(kw).value)
            if aksi == "Z": ds.data_element(kw).value = ""
            elif aksi == "X": delattr(ds, kw)
            elif aksi == "U":
                baru = generate_uid(); setattr(ds, kw, baru)
                if kw == "SOPInstanceUID": ds.file_meta.MediaStorageSOPInstanceUID = baru
            log.append({"keyword": kw, "aksi": aksi, "sebelum": lama[:40], "sesudah": (str(ds.data_element(kw).value)[:40] if kw in ds else "(dihapus)")})
        n_priv = sum(1 for el in ds if el.tag.is_private); ds.remove_private_tags()
        log.append({"keyword": "(private tags)", "aksi": "X", "sebelum": f"{n_priv} elemen", "sesudah": "(dihapus)"})
        if kotak_teks is not None:
            arr = ds.pixel_array.copy(); x0, y0, x1, y1 = kotak_teks; arr[y0:y1 + 1, x0:x1 + 1] = 0
            ds.PixelData = arr.astype(np.uint16).tobytes()
            log.append({"keyword": "PixelData (wilayah teks)", "aksi": "tutup", "sebelum": "teks identitas FIKTIF", "sesudah": "ditutup hitam"})
            ds.BurnedInAnnotation = "NO"
        ds.PatientIdentityRemoved = "YES"
        ds.DeidentificationMethod = "Subset PS3.15 Annex E Basic Profile (demo buku)"
        kode = Dataset(); kode.CodeValue = "113100"; kode.CodingSchemeDesignator = "DCM"; kode.CodeMeaning = "Basic Application Confidentiality Profile"
        ds.DeidentificationMethodCodeSequence = [kode]
        return ds, pd.DataFrame(log)
    ds_anon, T26 = anonimkan(ds, KOTAK)
    p_anon = DIR_DCM / "fiktif_sesudah.dcm"; ds_anon.save_as(p_anon)
    T26.to_csv(OUT / "tabel_2_6_anonimisasi.csv", index=False)
    # verifikasi: tidak ada string FIKTIF tersisa di header
    sisa = [el.keyword for el in pydicom.dcmread(p_anon) if el.keyword != "PixelData" and "FIKTIF" in str(el.value)]
    print("Elemen header yang masih memuat 'FIKTIF':", sisa or "tidak ada")
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(pydicom.dcmread(p_phi).pixel_array, cmap="gray"); ax[0].set_title("Sebelum: teks identitas FIKTIF tertanam")
    ax[1].imshow(ds_anon.pixel_array, cmap="gray"); ax[1].set_title("Sesudah: wilayah teks ditutup")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "gambar_2_3_anonimisasi.png", bbox_inches="tight"); plt.show()
ANON_SISA = sisa
T26

## 2.7 Kebocoran data: pembagian acak per sel vs per pasien (NIH Malaria)
Dataset NIH *Malaria Cell Images* berisi 27.558 citra sel (13.779 *Parasitized*, 13.779 *Uninfected*) dari apusan darah tipis.
Awalan nama berkas (mis. `C100P61ThinF_...`) diperlakukan sebagai **kode kelompok/pasien** — ini **asumsi** dari pola nama berkas
yang dicocokkan dengan deskripsi dataset (150 pasien terinfeksi + 50 sehat); hasil hitungan di bawah dipakai untuk mengecek asumsi itu.

Rancangan:
1. Sisihkan **20 % kelompok** sebagai **set eksternal** ("pasien baru") — tidak pernah dipakai melatih.
2. Pada 80 % sisanya, bandingkan dua cara membagi latih/uji internal (80/20):
   **A. acak per sel** (sel dari pasien yang sama bisa masuk latih dan uji) dan **B. per kelompok** (satu pasien hanya di satu sisi).
3. CNN kecil yang sama (64 px), epoch tetap, beberapa seed. Bandingkan skor uji internal vs eksternal.
Kebocoran terlihat bila skema A memberi skor internal yang lebih optimistis daripada kinerjanya pada pasien baru.

In [ ]:
import re, zipfile
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import GroupShuffleSplit, train_test_split
with catat_waktu("6_malaria_siapkan"):
    pastikan_dataset("malaria")
    akar = ekstrak_lokal("malaria", "cell_images.zip")
    berkas = []
    for kelas, lab in [("Parasitized", 1), ("Uninfected", 0)]:
        for p in sorted(akar.rglob(f"{kelas}/*.png")): berkas.append((p, lab, re.match(r"^(C\d+)", p.name).group(1)))
    DF = pd.DataFrame(berkas, columns=["path", "label", "kelompok"])
    if SMOKE:
        kel = sorted(DF.kelompok.unique())[:30]; DF = DF[DF.kelompok.isin(kel)].groupby("kelompok").head(20).reset_index(drop=True)
    UK = 64
    def baca(p): return np.asarray(Image.open(p).convert("RGB").resize((UK, UK), Image.BILINEAR), dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex: XM = np.stack(list(ex.map(baca, DF.path)))
    YM = DF.label.values.astype(np.int64); GM = DF.kelompok.values
per = DF.groupby("kelompok").label.agg(["count", "sum"])
T27 = pd.DataFrame([
    {"besaran": "jumlah citra sel", "nilai": len(DF)},
    {"besaran": "Parasitized / Uninfected", "nilai": f"{int(YM.sum())} / {int((YM == 0).sum())}"},
    {"besaran": "jumlah kelompok (awalan C<n>)", "nilai": per.shape[0]},
    {"besaran": "kelompok berisi kedua kelas", "nilai": int(((per["sum"] > 0) & (per["sum"] < per["count"])).sum())},
    {"besaran": "kelompok hanya Uninfected", "nilai": int((per["sum"] == 0).sum())},
    {"besaran": "kelompok hanya Parasitized", "nilai": int((per["sum"] == per["count"]).sum())},
    {"besaran": "sel per kelompok: min / median / maks", "nilai": f"{per['count'].min()} / {per['count'].median():.0f} / {per['count'].max()}"}])
T27.to_csv(OUT / "tabel_2_7_malaria_kelompok.csv", index=False)
rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(2, 8, figsize=(12, 3.4))
for r, lab in enumerate([1, 0]):
    for c, i in enumerate(rng.choice(np.where(YM == lab)[0], 8, replace=False)):
        ax[r, c].imshow(XM[i]); ax[r, c].axis("off")
    ax[r, 0].set_title(["Uninfected", "Parasitized"][lab], loc="left", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "gambar_2_4_malaria_contoh.png", bbox_inches="tight"); plt.show()
print("bentuk XM:", XM.shape); T27

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_dalam, idx_eks = next(gss.split(XM, YM, GM))
def bagi(skema, seed):
    if skema == "A_acak_per_sel":
        tr, te = train_test_split(idx_dalam, test_size=0.2, random_state=seed, stratify=YM[idx_dalam])
    else:
        g = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        a, b = next(g.split(idx_dalam, YM[idx_dalam], GM[idx_dalam])); tr, te = idx_dalam[a], idx_dalam[b]
    return tr, te

class CNNKecil(nn.Module):
    def __init__(self):
        super().__init__()
        def blok(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.fitur = nn.Sequential(blok(3, 32), blok(32, 64), blok(64, 128), blok(128, 128))
        self.kepala = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x): return self.kepala(self.fitur(x)).squeeze(1)

XT = torch.tensor(XM).permute(0, 3, 1, 2).contiguous()          # uint8 (N,3,64,64) di CPU
def batch_ke_dev(idx):
    return XT[idx].to(DEV, non_blocking=True).float().div(255).sub(0.5).div(0.5)
@torch.no_grad()
def skor(model, idx, bs=1024):
    model.eval(); out = []
    for i in range(0, len(idx), bs): out.append(torch.sigmoid(model(batch_ke_dev(idx[i:i + bs]))).float().cpu())
    return torch.cat(out).numpy()
def latih(tr, seed, epoch, bs=256):
    seed_semua(seed); model = CNNKecil().to(DEV); opt = torch.optim.Adam(model.parameters(), 1e-3)
    yt = torch.tensor(YM, dtype=torch.float32); g = torch.Generator().manual_seed(seed)
    for ep in range(epoch):
        model.train(); perm = tr[torch.randperm(len(tr), generator=g).numpy()]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]; x = batch_ke_dev(b)
            if torch.rand(1, generator=g).item() < 0.5: x = x.flip(3)
            loss = F.binary_cross_entropy_with_logits(model(x), yt[b].to(DEV))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return model
def ukur(y, p):
    return {"AUC": roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan"), "akurasi": accuracy_score(y, p >= 0.5)}

EPOCH = 2 if SMOKE else 8
SEEDS = [0] if SMOKE else [0, 1, 2]
hasil = []
with catat_waktu("7_malaria_latih_kebocoran"):
    for skema in ["A_acak_per_sel", "B_per_kelompok"]:
        for s in SEEDS:
            tr, te = bagi(skema, s); t0 = time.perf_counter()
            m = latih(tr, s, EPOCH); dt = time.perf_counter() - t0
            irisan_kel = len(set(GM[tr]) & set(GM[te]))
            ri, re_ = ukur(YM[te], skor(m, te)), ukur(YM[idx_eks], skor(m, idx_eks))
            hasil.append({"skema": skema, "seed": s, "n_latih": len(tr), "n_uji_internal": len(te), "n_eksternal": len(idx_eks),
                          "kelompok_uji_yang_juga_di_latih": irisan_kel, "kelompok_uji_internal": len(set(GM[te])),
                          "AUC_internal": ri["AUC"], "AUC_eksternal": re_["AUC"], "akurasi_internal": ri["akurasi"],
                          "akurasi_eksternal": re_["akurasi"], "detik_latih": dt})
            print(hasil[-1])
T28 = pd.DataFrame(hasil); T28["selisih_akurasi_internal_minus_eksternal"] = T28.akurasi_internal - T28.akurasi_eksternal
T28.to_csv(OUT / "tabel_2_8_kebocoran_per_seed.csv", index=False)
T29 = T28.groupby("skema")[["AUC_internal", "AUC_eksternal", "akurasi_internal", "akurasi_eksternal",
                             "selisih_akurasi_internal_minus_eksternal", "kelompok_uji_yang_juga_di_latih"]].agg(["mean", "std"])
T29.columns = ["_".join(c) for c in T29.columns]; T29.to_csv(OUT / "tabel_2_9_kebocoran_ringkas.csv")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, met in zip(ax, ["akurasi", "AUC"]):
    xs = np.arange(2); w = 0.35
    for j, (bag, warna) in enumerate([("internal", "#4c72b0"), ("eksternal", "#dd8452")]):
        mu = [T28[T28.skema == s][f"{met}_{bag}"].mean() for s in ["A_acak_per_sel", "B_per_kelompok"]]
        sd = [T28[T28.skema == s][f"{met}_{bag}"].std() if len(SEEDS) > 1 else 0 for s in ["A_acak_per_sel", "B_per_kelompok"]]
        a.bar(xs + (j - 0.5) * w, mu, w, yerr=sd, capsize=3, label=f"uji {bag}", color=warna)
        for x, m_ in zip(xs + (j - 0.5) * w, mu): a.text(x, m_, f"{m_:.3f}", ha="center", va="bottom", fontsize=7)
    a.set_xticks(xs); a.set_xticklabels(["A: acak per sel", "B: per kelompok"]); a.set_title(met)
    lo = min(T28[f"{met}_internal"].min(), T28[f"{met}_eksternal"].min()); a.set_ylim(max(0, lo - 0.05), 1.0)
ax[0].legend(loc="lower right"); plt.suptitle(f"Kebocoran data: rerata ± SD {len(SEEDS)} seed, {EPOCH} epoch")
plt.tight_layout(); plt.savefig(OUT / "gambar_2_5_kebocoran.png", bbox_inches="tight"); plt.show()
T29

## 2.8 Reproduktibilitas
Model skema B (seed 0) dilatih **dua kali** dengan seed dan pengaturan deterministik yang sama; bobot dan skor dibandingkan.
Lalu dicatat versi paket dan dibuat **kartu model** ringkas berisi angka dari jalannya notebook ini.

In [ ]:
with catat_waktu("8_reproduktibilitas"):
    tr, te = bagi("B_per_kelompok", 0); ep_r = 1 if SMOKE else 2
    m1 = latih(tr, 0, ep_r); m2 = latih(tr, 0, ep_r)
    s1, s2 = skor(m1, te), skor(m2, te)
    beda_bobot = max(float((a - b).abs().max()) for a, b in zip(m1.state_dict().values(), m2.state_dict().values()) if a.is_floating_point())
    m3 = latih(tr, 1, ep_r); s3 = skor(m3, te)
REPRO = {"epoch": ep_r, "selisih_bobot_maks_seed_sama": beda_bobot, "selisih_skor_maks_seed_sama": float(np.abs(s1 - s2).max()),
         "identik_seed_sama": bool(beda_bobot == 0.0), "AUC_seed0_run1": ukur(YM[te], s1)["AUC"], "AUC_seed0_run2": ukur(YM[te], s2)["AUC"],
         "AUC_seed1": ukur(YM[te], s3)["AUC"], "selisih_skor_maks_seed_beda": float(np.abs(s1 - s3).max())}
pd.Series(REPRO).to_csv(OUT / "tabel_2_10_reproduktibilitas.csv", header=["nilai"])
(OUT / "requirements_bab02.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSI.items() if k not in ("gpu", "cuda", "python")) +
                                         f"\n# python {VERSI['python']} | CUDA {VERSI['cuda']} | GPU {VERSI['gpu']}\n")
rB = T28[T28.skema == "B_per_kelompok"]
kartu = f'''# Kartu Model — CNN kecil deteksi sel malaria (demo Bab 2)
- **Tujuan**: demonstrasi pendidikan tentang kebocoran data; **bukan** untuk penggunaan klinis.
- **Data**: NIH Malaria Cell Images (Rajaraman dkk., 2018), {len(DF)} citra sel, {per.shape[0]} kelompok (awalan nama berkas, diasumsikan pasien).
- **Pembagian**: 20 % kelompok disisihkan sebagai set eksternal; latih/uji internal per kelompok (skema B).
- **Arsitektur**: 4 blok Conv-BN-ReLU-MaxPool (32-64-128-128), GAP, dropout 0,3; masukan RGB 64×64; {sum(p.numel() for p in m1.parameters()):,} parameter.
- **Pelatihan**: Adam lr 1e-3, batch 256, {EPOCH} epoch, flip horizontal acak; seed {SEEDS}.
- **Kinerja (rerata seed, skema B)**: AUC internal {rB.AUC_internal.mean():.4f}, AUC eksternal {rB.AUC_eksternal.mean():.4f}, akurasi eksternal {rB.akurasi_eksternal.mean():.4f} (ambang 0,5).
- **Lingkungan**: {VERSI['gpu']}, torch {VERSI['torch']}, CUDA {VERSI['cuda']}, Python {VERSI['python']}.
- **Reproduktibilitas**: dua kali latih seed sama → selisih bobot maks {beda_bobot:.3g}.
- **Keterbatasan**: satu sumber data; asumsi kelompok=pasien dari nama berkas; tanpa kalibrasi; tanpa validasi klinis.
'''
(OUT / "kartu_model_malaria.md").write_text(kartu); print(kartu); REPRO

## 2.9 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(t21=T21.to_dict("records"), anonimisasi_sisa=ANON_SISA, malaria=T27.to_dict("records"), kebocoran=T28.to_dict("records"), reproduktibilitas=REPRO, kelompok_eksternal=int(len(set(GM[idx_eks]))), sel_eksternal=int(len(idx_eks))))))
(OUT / "ringkasan_bab02.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass